In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import jax
import jax.numpy as jnp
from ase.visualize import view
from ase.atoms import Atoms

from msmjax.core.shortrange import make_eval_pair_pot, _gen_supercell
from msmjax.utils.benchmarking import (
    eval_lammps_pppm,
    path_input_structures,
)

LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

# Function definitions

In [2]:
def coulomb_kernel(r):
    return 1.0 / r


def calc_nonperiodic_ref_energy(positions, charges):
    n_dim = positions.shape[1]
    compute_pair_term = make_eval_pair_pot(
        kernel_fn=coulomb_kernel, pbc=(False,) * n_dim
    )
    return compute_pair_term(positions, charges)


def calc_nonperiodic_ref_forces(positions, charges):
    return -jax.grad(calc_nonperiodic_ref_energy, argnums=0)(
        positions, charges
    )


def calc_nonperiodic_ref_chargegrad(positions, charges):
    return jax.grad(calc_nonperiodic_ref_energy, argnums=1)(positions, charges)


def calc_energy_from_scaled(scaled_positions, charges, cell):
    positions = scaled_positions @ cell
    return calc_nonperiodic_ref_energy(positions, charges)


def calc_stress(positions, charges, cell):
    n_dim = positions.shape[1]
    scaled_positions = jnp.linalg.solve(cell.T, positions.T).T

    def deformation_energy(epsilon):
        return calc_energy_from_scaled(
            scaled_positions,
            charges,
            cell @ (jnp.eye(n_dim) + 0.5 * (epsilon + epsilon.T)),
        )

    return jax.grad(deformation_energy)(jnp.zeros_like(cell)) / jnp.fabs(
        jnp.linalg.det(cell)
    )


inds_matrix_to_six_component_stress = (
    jnp.array([0, 1, 2, 0, 0, 1]),
    jnp.array([0, 1, 2, 1, 2, 2]),
)


def calc_nonperiodic_reference_results(positions, charges, cell):
    # This is less likely to run out of memory than calculating everything
    # with a single jax.value_and_grad call
    value = jax.jit(calc_nonperiodic_ref_energy)(positions, charges)
    forces = jax.jit(calc_nonperiodic_ref_forces)(positions, charges)
    chargegrad = jax.jit(calc_nonperiodic_ref_chargegrad)(positions, charges)
    stress = jax.jit(calc_stress)(positions, charges, cell)
    return value, forces, chargegrad, stress

In [3]:
def load_one_structure(n_particles):
    structures = onp.load(
        path_input_structures / ("structures_" + str(n_particles) + ".npz")
    )
    # TODO: Also test different structures of the same number of particles?
    #  (i.e., other values for idx_structure than 0)
    idx_structure = 0
    pos = structures["positions"][idx_structure]
    chg = structures["charges"][idx_structure]
    cell = structures["cells"][idx_structure]
    pos = pos.astype(onp.float64)
    chg = chg.astype(onp.float64)
    cell = cell.astype(onp.float64)
    return pos, chg, cell

# Non-periodic

## Cubic

In [4]:
(pos, chg, cell) = load_one_structure(7000)

print(cell)

e_ref, f_ref, chargegrad_ref, stress_ref = calc_nonperiodic_reference_results(
    pos, chg, cell
)

fname = "nonperiodic_cubic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref[inds_matrix_to_six_component_stress],
)

[[19.12931252  0.          0.        ]
 [ 0.         19.12931252  0.        ]
 [ 0.          0.         19.12931252]]


## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the minimum number of grid points is reached along some axis before the others.

In [5]:
(pos, chg, cell) = load_one_structure(1000)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

print(cell)

e_ref, f_ref, chargegrad_ref, stress_ref = calc_nonperiodic_reference_results(
    pos, chg, cell
)

fname = "nonperiodic_ortho-different-sidelengths.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref[inds_matrix_to_six_component_stress],
)

atoms = Atoms(positions=pos, cell=cell, charges=chg)
view(atoms)

[[33.  0.  0.]
 [ 0. 20.  0.]
 [ 0.  0.  9.]]


<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

## Triclinic

In [6]:
(pos, chg, cell) = load_one_structure(7000)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.8, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

print(cell)

e_ref, f_ref, chargegrad_ref, stress_ref = calc_nonperiodic_reference_results(
    pos, chg, cell
)

fname = "nonperiodic_triclinic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref[inds_matrix_to_six_component_stress],
)

view(atoms)

[[15.30345001  0.          0.        ]
 [-9.56465626 16.5664706   0.        ]
 [ 0.          7.14619683 22.81881743]]


<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

# Periodic

## Cubic

In [7]:
(pos, chg, cell) = load_one_structure(500)

e_ref, f_ref, chargegrad_ref, stress_ref = eval_lammps_pppm(
    pos, chg, cell, LAMMPS_EXECUTABLE, max_neighbors_one_atom=10000
)

fname = "periodic_cubic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref,
)

## Orthorhombic with different side lengths

The intention is that the side lengths are so different that the grid is reduced to a single point along some axis faster than along the others.

In [8]:
(pos, chg, cell) = load_one_structure(100)
(pos, chg, cell) = _gen_supercell(pos, chg, cell, supercell_diag=(3, 2, 1))
scaled_pos = pos @ onp.linalg.pinv(cell)
axis_stretch_factors = onp.array([1.1, 1.0, 0.9])
cell *= axis_stretch_factors
pos = scaled_pos @ cell

e_ref, f_ref, chargegrad_ref, stress_ref = eval_lammps_pppm(
    pos,
    chg,
    cell,
    LAMMPS_EXECUTABLE,
    max_neighbors_one_atom=10000,
    show_stdout=True,
)

fname = "periodic_ortho-different-sidelengths.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref,
)

atoms = Atoms(positions=pos, cell=cell, charges=chg)
view(atoms)

LAMMPS (27 Jun 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Reading data file ...
  orthogonal box = (0 0 0) to (15.317243 9.2831774 4.1774298)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  600 atoms
  read_data CPU = 0.003 seconds

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Your simulation uses code contributions which should be cited:
- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419
The log file lists these citations in BibTeX format.

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

PPPM initialization ...
  using 12-bit tables for long-range coulomb (src/kspace.cpp:342)
  G vector (1/distance) = 0.32945224
  grid = 15 12 8
  stencil order = 5
  estimated absolute RMS force accuracy = 9.1086587e-05
  estimated relative force accuracy = 6.3256134e-06
  using double precision KISS FFT
  3d grid and FFT values/proc = 7106 1440
Generated 0 o

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>

## Triclinic

In [9]:
(pos, chg, cell) = load_one_structure(500)

atoms = Atoms(positions=pos, charges=chg, cell=cell)
new_lengths = onp.diag(cell) * (0.9, 1.0, 1.25)
new_angles = [75, 90, 120]
nonortho_cell = onp.concatenate([new_lengths, new_angles])
atoms.set_cell(nonortho_cell, scale_atoms=True)
pos = onp.array(atoms.get_positions())
chg = onp.array(atoms.get_initial_charges())
cell = onp.array(atoms.get_cell()[...])

e_ref, f_ref, chargegrad_ref, stress_ref = eval_lammps_pppm(
    pos,
    chg,
    cell,
    LAMMPS_EXECUTABLE,
    max_neighbors_one_atom=10000,
    show_stdout=True,
)

fname = "periodic_triclinic.npz"
onp.savez_compressed(
    fname,
    positions=pos,
    charges=chg,
    cell=cell,
    energy=e_ref,
    forces=f_ref,
    charge_gradient=chargegrad_ref,
    stress=stress_ref,
)

view(atoms)

LAMMPS (27 Jun 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Reading data file ...
  triclinic box = (0 0 0) to (7.1433045 6.873648 9.4678295) with tilt (-3.9685025 0 2.9650517)
  1 by 1 by 1 MPI processor grid
  reading atoms ...
  500 atoms
  read_data CPU = 0.004 seconds

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

Your simulation uses code contributions which should be cited:
- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419
The log file lists these citations in BibTeX format.

CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE

PPPM initialization ...
  using 12-bit tables for long-range coulomb (src/kspace.cpp:342)
  G vector (1/distance) = 0.29278275
  grid = 10 15 15
  stencil order = 5
  estimated absolute RMS force accuracy = 0.00084001054
  estimated relative force accuracy = 5.8335504e-05
  using double precision KISS FFT
  3d grid and FFT val

<Popen: returncode: None args: ['/home/florian/anaconda3/envs/msmjax_py3.11/...>